In [0]:
orders = spark.table("default.olist_orders_dataset")

items = spark.table("default.olist_order_items_dataset")

payments = spark.table("default.olist_order_payments_dataset")

reviews = spark.table("default.olist_order_reviews_dataset")

products = spark.table("default.olist_products_dataset_v2")

In [0]:
print("Orders:", orders.count())
print("Items:", items.count())
print("Payments:", payments.count())
print("Reviews:", reviews.count())
print("Products:", products.count())

Orders: 99441
Items: 112650
Payments: 103886
Reviews: 104162
Products: 32951


In [0]:
display(
    products.select(
        "product_id",
        "product_category_name"
    ).limit(10)
)

product_id,product_category_name
1e9e8ef04dbcff4541ed26657ea517e5,perfumaria
3aa071139cb16b67ca9e5dea641aaa2f,artes
96bd76ec8810374ed1b65e291975717f,esporte_lazer
cef67bcfe19066a932b7673e239eb23d,bebes
9dc1a7de274444849c219cff195d0b71,utilidades_domesticas
41d3672d4792049fa1779bb35283ed13,instrumentos_musicais
732bd381ad09e530fe0a5f457d81becb,cool_stuff
2548af3e6e77a690cf3eb6368e9ab61e,moveis_decoracao
37cc742be07708b53a98702e77a21a02,eletrodomesticos
8c92109888e8cdf9d66dc7e463025574,brinquedos


In [0]:
from pyspark.sql.functions import col

business_df = (
    items.alias("i")
    .join(
        payments.alias("p"),
        col("i.order_id") == col("p.order_id"),
        "inner"
    )
    .join(
        reviews.alias("r"),
        col("i.order_id") == col("r.order_id"),
        "left"
    )
    .join(
        products.alias("pr"),
        col("i.product_id") == col("pr.product_id"),
        "inner"
    )
    .select(
        col("i.order_id").alias("order_id"),
        col("i.product_id").alias("product_id"),
        col("pr.product_category_name").alias("category"),
        col("p.payment_value").alias("revenue"),
        col("r.review_score").alias("review_score")
    )
)

In [0]:
display(business_df.limit(10))

order_id,product_id,category,revenue,review_score
00010242fe8c5a6d1ba2dd792cb16214,4244733e06e7ecb4970a6e2683c13e61,cool_stuff,72.19,5
00018f77f2f0320c557190d7a144bdd3,e5f2d52b802189ee658865ca93d83a8f,pet_shop,259.83,4
000229ec398224ef6ca0657da4fc703e,c777355d18b72b67abbeef9df44fd0fd,moveis_decoracao,216.87,5
00024acbcdf0a6daa1e931b038114c75,7634da152a4610f1595efa32f14722fc,perfumaria,25.78,4
00042b26cf59d7ce69dfabb4e55b4fd9,ac6c3623068f30de03045865e4e10089,ferramentas_jardim,218.04,5
00048cc3ae777c65dbb7d2a0634bc1ea,ef92defde845ab8450f9d70c526ef70f,utilidades_domesticas,34.59,4
00054e8431b9d7675808bcb819fb4a32,8d4f2bb7e93e6710a28f34fa83ee7d28,telefonia,31.75,4
000576fe39319847cbb9d288c5617fa6,557d850972a7d6f792fd18ae1400d9b6,ferramentas_jardim,880.75,5
0005a1a1728c9d785b8e2b08b904576c,310ae3c140ff94b03219ad0adc3c778f,beleza_saude,157.6,1
0005f50442cb953dcd1d21e1fb923495,4535b0e1091c278dfd193e5a1d63b39f,livros_tecnicos,65.39,4


In [0]:
business_df.filter(
    col("category").isNull()
).count()

1709

In [0]:
from pyspark.sql.functions import (
    col,
    avg,
    sum,
    count,
    when
)

In [0]:
category_metrics = (
    business_df
    .filter(col("category").isNotNull())
    .groupBy("category")
    .agg(
        avg("review_score").alias("avg_review_score"),
        sum("revenue").alias("revenue"),
        count("order_id").alias("total_orders")
    )
)

In [0]:
print("Categories:", category_metrics.count())

Categories: 73


In [0]:
category_metrics.orderBy(col("revenue").desc()).show(10, False)

+----------------------+------------------+------------------+------------+
|category              |avg_review_score  |revenue           |total_orders|
+----------------------+------------------+------------------+------------+
|cama_mesa_banho       |3.890605216510509 |1743998.8000000045|11988       |
|beleza_saude          |4.137972646822204 |1662963.5900000036|10029       |
|informatica_acessorios|3.9360888340530535|1599481.0600000008|8150        |
|moveis_decoracao      |3.912158298067025 |1443963.6100000034|8832        |
|relogios_presentes    |4.017691933127739 |1430553.4800000067|6213        |
|esporte_lazer         |4.107470364571684 |1400223.0700000043|9004        |
|utilidades_domesticas |4.060428318101214 |1097900.0899999966|7380        |
|automotivo            |4.064279155188246 |855095.6799999996 |4400        |
|ferramentas_jardim    |4.023913997367266 |840721.5900000007 |4590        |
|cool_stuff            |4.140766902119072 |781933.9699999997 |3999        |
+-----------

In [0]:
from pyspark.sql.functions import when, col

category_metrics = category_metrics.withColumn(
    "risk_score",
    when(
        (col("avg_review_score") < 3.8) &
        (col("revenue") > 100000),
        90
    )
    .when(
        col("avg_review_score") < 4.2,
        60
    )
    .otherwise(20)
)

In [0]:
category_metrics = category_metrics.withColumn(
    "risk_level",
    when(col("risk_score") >= 80, "High Risk")
    .when(col("risk_score") >= 50, "Medium Risk")
    .otherwise("Low Risk")
)

In [0]:
display(
    category_metrics.groupBy("risk_level")
    .count()
)

risk_level,count
Medium Risk,55
High Risk,2
Low Risk,16


In [0]:
category_metrics = category_metrics.withColumn(
    "revenue_at_risk",
    col("revenue") *
    (col("risk_score") / 100)
)

In [0]:
display(
    category_metrics.select(
        "category",
        "revenue",
        "risk_score",
        "revenue_at_risk"
    )
)

category,revenue,risk_score,revenue_at_risk
construcao_ferramentas_construcao,243218.5100000002,60,145931.10600000012
industria_comercio_e_negocios,56842.77999999998,60,34105.66799999998
telefonia_fixa,207071.0499999999,90,186363.94499999992
fashion_esporte,3685.0100000000007,20,737.0020000000002
eletronicos,260021.2700000009,60,156012.76200000054
fashion_underwear_e_moda_praia,12714.539999999995,60,7628.7239999999965
eletrodomesticos_2,124865.92000000003,20,24973.18400000001
portateis_cozinha_e_preparadores_de_alimentos,4335.65,60,2601.39
construcao_ferramentas_ferramentas,21069.07,20,4213.814
moveis_escritorio,652016.4999999998,90,586814.8499999999


In [0]:
category_metrics = category_metrics.withColumn(
    "recommended_action",
    when(
        col("risk_score") >= 80,
        "Immediate Quality Intervention"
    )
    .when(
        col("risk_score") >= 50,
        "Customer Retention Campaign"
    )
    .otherwise(
        "Monitor and Grow"
    )
)

In [0]:
display(
    category_metrics.groupBy(
        "recommended_action"
    ).count()
)

recommended_action,count
Customer Retention Campaign,55
Immediate Quality Intervention,2
Monitor and Grow,16


In [0]:
category_metrics = category_metrics.withColumn(
    "growth_opportunity",
    when(
        (col("avg_review_score") >= 4.3) &
        (col("total_orders") >= 500),
        "High Opportunity"
    )
    .when(
        col("avg_review_score") >= 4.0,
        "Medium Opportunity"
    )
    .otherwise(
        "Low Opportunity"
    )
)

In [0]:
display(
    category_metrics.groupBy(
        "growth_opportunity"
    ).count()
)

growth_opportunity,count
Medium Opportunity,47
Low Opportunity,25
High Opportunity,1


In [0]:
display(
    category_metrics.orderBy(
        col("revenue_at_risk").desc()
    )
)

category,avg_review_score,revenue,total_orders,risk_score,risk_level,revenue_at_risk,recommended_action,growth_opportunity
cama_mesa_banho,3.890605216510509,1743998.8000000045,11988,60,Medium Risk,1046399.2800000026,Customer Retention Campaign,Low Opportunity
beleza_saude,4.137972646822204,1662963.5900000036,10029,60,Medium Risk,997778.1540000021,Customer Retention Campaign,Medium Opportunity
informatica_acessorios,3.9360888340530535,1599481.0600000008,8150,60,Medium Risk,959688.6360000004,Customer Retention Campaign,Low Opportunity
moveis_decoracao,3.912158298067025,1443963.6100000034,8832,60,Medium Risk,866378.166000002,Customer Retention Campaign,Low Opportunity
relogios_presentes,4.017691933127739,1430553.4800000067,6213,60,Medium Risk,858332.0880000041,Customer Retention Campaign,Medium Opportunity
esporte_lazer,4.107470364571684,1400223.0700000043,9004,60,Medium Risk,840133.8420000025,Customer Retention Campaign,Medium Opportunity
utilidades_domesticas,4.060428318101214,1097900.0899999966,7380,60,Medium Risk,658740.0539999979,Customer Retention Campaign,Medium Opportunity
moveis_escritorio,3.526790750141004,652016.4999999998,1788,90,High Risk,586814.8499999999,Immediate Quality Intervention,Low Opportunity
automotivo,4.064279155188246,855095.6799999996,4400,60,Medium Risk,513057.4079999997,Customer Retention Campaign,Medium Opportunity
ferramentas_jardim,4.023913997367266,840721.5900000007,4590,60,Medium Risk,504432.9540000004,Customer Retention Campaign,Medium Opportunity


In [0]:
display(
    category_metrics.groupBy("risk_level")
    .count()
)

risk_level,count
Medium Risk,55
High Risk,2
Low Risk,16


In [0]:
display(
    category_metrics.groupBy("recommended_action")
    .count()
)

recommended_action,count
Customer Retention Campaign,55
Immediate Quality Intervention,2
Monitor and Grow,16


In [0]:
display(
    category_metrics.groupBy("growth_opportunity")
    .count()
)

growth_opportunity,count
Medium Opportunity,47
Low Opportunity,25
High Opportunity,1


In [0]:
category_metrics = category_metrics.withColumn(
    "growth_opportunity",
    when(
        (col("avg_review_score") >= 4.3) &
        (col("total_orders") >= 500),
        "High Opportunity"
    )
    .when(
        col("avg_review_score") >= 4.0,
        "Medium Opportunity"
    )
    .otherwise(
        "Low Opportunity"
    )
)

In [0]:
display(
    category_metrics.groupBy(
        "growth_opportunity"
    ).count()
)

growth_opportunity,count
Medium Opportunity,47
Low Opportunity,25
High Opportunity,1


In [0]:
display(
    category_metrics.selectExpr(
        "round(sum(revenue),2) as Total_Revenue"
    )
)

Total_Revenue
2.016311522E7


In [0]:
display(
    category_metrics.selectExpr(
        "round(sum(revenue_at_risk),2) as Revenue_At_Risk"
    )
)

Revenue_At_Risk
1.211094687E7


In [0]:
display(
    category_metrics.groupBy("risk_level")
    .count()
)

risk_level,count
Medium Risk,55
High Risk,2
Low Risk,16


Databricks visualization. Run in Databricks to view.

In [0]:
display(
    category_metrics.select(
        "category",
        "revenue_at_risk"
    )
)

category,revenue_at_risk
construcao_ferramentas_construcao,145931.10600000012
industria_comercio_e_negocios,34105.66799999998
telefonia_fixa,186363.94499999992
fashion_esporte,737.0020000000002
eletronicos,156012.76200000054
fashion_underwear_e_moda_praia,7628.7239999999965
eletrodomesticos_2,24973.18400000001
portateis_cozinha_e_preparadores_de_alimentos,2601.39
construcao_ferramentas_ferramentas,4213.814
moveis_escritorio,586814.8499999999


Databricks visualization. Run in Databricks to view.

In [0]:
display(
    category_metrics.groupBy(
        "recommended_action"
    ).count()
)

recommended_action,count
Customer Retention Campaign,55
Immediate Quality Intervention,2
Monitor and Grow,16


Databricks visualization. Run in Databricks to view.

In [0]:
display(
    category_metrics.groupBy(
        "growth_opportunity"
    ).count()
)

growth_opportunity,count
Medium Opportunity,47
Low Opportunity,25
High Opportunity,1


Databricks visualization. Run in Databricks to view.

In [0]:
display(
    category_metrics.orderBy(
        col("revenue_at_risk").desc()
    )
)

category,avg_review_score,revenue,total_orders,risk_score,risk_level,revenue_at_risk,recommended_action,growth_opportunity
cama_mesa_banho,3.890605216510509,1743998.8000000045,11988,60,Medium Risk,1046399.2800000026,Customer Retention Campaign,Low Opportunity
beleza_saude,4.137972646822204,1662963.5900000036,10029,60,Medium Risk,997778.1540000021,Customer Retention Campaign,Medium Opportunity
informatica_acessorios,3.9360888340530535,1599481.0600000008,8150,60,Medium Risk,959688.6360000004,Customer Retention Campaign,Low Opportunity
moveis_decoracao,3.912158298067025,1443963.6100000034,8832,60,Medium Risk,866378.166000002,Customer Retention Campaign,Low Opportunity
relogios_presentes,4.017691933127739,1430553.4800000067,6213,60,Medium Risk,858332.0880000041,Customer Retention Campaign,Medium Opportunity
esporte_lazer,4.107470364571684,1400223.0700000043,9004,60,Medium Risk,840133.8420000025,Customer Retention Campaign,Medium Opportunity
utilidades_domesticas,4.060428318101214,1097900.0899999966,7380,60,Medium Risk,658740.0539999979,Customer Retention Campaign,Medium Opportunity
moveis_escritorio,3.526790750141004,652016.4999999998,1788,90,High Risk,586814.8499999999,Immediate Quality Intervention,Low Opportunity
automotivo,4.064279155188246,855095.6799999996,4400,60,Medium Risk,513057.4079999997,Customer Retention Campaign,Medium Opportunity
ferramentas_jardim,4.023913997367266,840721.5900000007,4590,60,Medium Risk,504432.9540000004,Customer Retention Campaign,Medium Opportunity


Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.functions import col

top_risk = (
    category_metrics
    .orderBy(col("revenue_at_risk").desc())
    .limit(10)
    .toPandas()
)

business_summary = top_risk.to_string(index=False)

print(business_summary)

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-8297849334587776>, line 4
      1 from pyspark.sql.functions import col
      3 top_risk = (
----> 4     df
      5     .orderBy(col("revenue_at_risk").desc())
      6     .limit(10)
      7     .toPandas()
      8 )
     10 business_summary = top_risk.to_string(index=False)
     12 print(business_summary)

NameError: name 'df' is not defined

In [0]:
from pyspark.sql.functions import col

top_risk = (
    category_metrics
    .orderBy(col("revenue_at_risk").desc())
    .limit(10)
    .toPandas()
)

business_summary = top_risk.to_string(index=False)

print(business_summary)

              category  avg_review_score    revenue  total_orders  risk_score  risk_level  revenue_at_risk             recommended_action growth_opportunity
       cama_mesa_banho          3.890605 1743998.80         11988          60 Medium Risk      1046399.280    Customer Retention Campaign    Low Opportunity
          beleza_saude          4.137973 1662963.59         10029          60 Medium Risk       997778.154    Customer Retention Campaign Medium Opportunity
informatica_acessorios          3.936089 1599481.06          8150          60 Medium Risk       959688.636    Customer Retention Campaign    Low Opportunity
      moveis_decoracao          3.912158 1443963.61          8832          60 Medium Risk       866378.166    Customer Retention Campaign    Low Opportunity
    relogios_presentes          4.017692 1430553.48          6213          60 Medium Risk       858332.088    Customer Retention Campaign Medium Opportunity
         esporte_lazer          4.107470 1400223.07       

In [0]:
def ask_strategy_agent(question):

    prompt = f"""
    You are a Chief Ecommerce Strategy Officer.

    Business Data:

    {business_summary}

    Question:
    {question}

    Provide:

    1. Executive Summary
    2. Key Risks
    3. Recommended Actions
    4. Expected Business Impact

    Keep the response concise and professional.
    """

    response = model.generate_content(prompt)

    return response.text

In [0]:
print(
    ask_strategy_agent(
        "Which category needs immediate attention?"
    )
)

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-8297849334587779>, line 2
      1 print(
----> 2     ask_strategy_agent(
      3         "Which category needs immediate attention?"
      4     )
      5 )

File <command-8297849334587778>, line 23, in ask_strategy_agent(question)
      1 def ask_strategy_agent(question):
      3     prompt = f"""
      4     You are a Chief Ecommerce Strategy Officer.
      5 
   (...)
     20     Keep the response concise and professional.
     21     """
---> 23     response = model.generate_content(prompt)
     25     return response.text

NameError: name 'model' is not defined

In [0]:
category_metrics.write.mode("overwrite").saveAsTable(
    "default.category_metrics_final"
)